In [ ]:
import numpy as np
import cv2
from joblib import Parallel, delayed
from scipy.signal import correlate2d
import pickle
import src.Functions as Fu
import src.Filter as Ft
from tqdm import tqdm
from scipy.fft import fft2, ifft2, fftshift

def correlacao_normalizada_bloco(bloco, padrao):
    # """Calcula a correlação normalizada entre um bloco e um padrão."""
    # bloco = bloco.astype(float)
    # padrao = padrao.astype(float)
    
    # bloco_norm = bloco - np.mean(bloco)
    # padrao_norm = padrao - np.mean(padrao)
    
    # correlacao = np.sum(bloco_norm * padrao_norm)
    # norma_bloco = np.sqrt(np.sum(bloco_norm**2))
    # norma_padrao = np.sqrt(np.sum(padrao_norm**2))
    
    # if norma_bloco == 0 or norma_padrao == 0:
    #     return 0
    # else:
    #     return correlacao / (norma_bloco * norma_padrao)
    """
    Calcula a correlação normalizada entre um bloco e um padrão usando FFT.
    
    :param bloco: Bloco da imagem
    :param padrao: Padrão para correlação
    :return: Valor de correlação normalizada
    """
    # Garante que bloco e padrão tenham o mesmo tamanho
    assert bloco.shape == padrao.shape, "Bloco e padrão devem ter o mesmo tamanho"
    
    # Converte para float e normaliza
    bloco = bloco.astype(float) - np.mean(bloco)
    padrao = padrao.astype(float) - np.mean(padrao)
    
    # Calcula as FFTs
    fft_bloco = fft2(bloco)
    fft_padrao = fft2(padrao)
    
    # Calcula a correlação cruzada no domínio da frequência
    correlacao = ifft2(fft_bloco * np.conj(fft_padrao)).real
    
    # Normalização
    norma_bloco = np.sqrt(np.sum(bloco**2))
    norma_padrao = np.sqrt(np.sum(padrao**2))
    correlacao=correlacao/ (norma_bloco * norma_padrao)
    
    if norma_bloco == 0 or norma_padrao == 0:
        return 0
    else:
        return correlacao[0][0]



def processar_bloco(Noisex, fingerprint, y_start, x_start, tamanho_bloco):
    y_end = y_start + tamanho_bloco
    x_end = x_start + tamanho_bloco
    bloco = Noisex[y_start:y_end, x_start:x_end]
    padrao = fingerprint[y_start:y_end, x_start:x_end]
    return correlacao_normalizada_bloco(bloco, padrao)

def gerar_mapa_correlacao_blocos_paralelo(Noisex, fingerprint, tamanho_bloco, k, n_jobs=-1):
    """
    Gera o mapa de correlação normalizada entre a imagem e o padrão usando blocos com sobreposição controlada,
    processando em paralelo.
    
    :param Noisex: Imagem de entrada
    :param fingerprint: Padrão para correlação (deve ter o mesmo tamanho que Noisex)
    :param tamanho_bloco: Tamanho do bloco (B x B)
    :param k: Sobreposição entre blocos (0 <= k < tamanho_bloco)
    :param n_jobs: Número de jobs para processamento paralelo (-1 para usar todos os cores)
    :return: Mapa de correlação normalizada
    """
    altura, largura = Noisex.shape
    passo = tamanho_bloco - k
    
    num_blocos_v = (altura - k) // passo
    num_blocos_h = (largura - k) // passo
    
    mapa_correlacao = np.zeros((altura, largura))
    
    # Prepara os argumentos para processamento paralelo
    args = [(Noisex, fingerprint, i * passo, j * passo, tamanho_bloco)
            for i in range(num_blocos_v)
            for j in range(num_blocos_h)]
    
    # Processa os blocos em paralelo
    resultados = Parallel(n_jobs=n_jobs)(
        delayed(processar_bloco)(*arg) for arg in tqdm(args, desc="Processando blocos", unit="bloco")
    )
    
    # Reconstrói o mapa de correlação
    for idx, (i, j) in enumerate([(i, j) for i in range(num_blocos_v) for j in range(num_blocos_h)]):
        y_start, x_start = i * passo, j * passo
        y_end, x_end = y_start + tamanho_bloco, x_start + tamanho_bloco
        mapa_correlacao[y_start:y_end, x_start:x_end] = resultados[idx]
    
    return mapa_correlacao


In [ ]:
with open('./DT.prnu', 'rb') as arquivo:
        Fingerprint = pickle.load(arquivo)

In [ ]:
questionado = 'DLT_1791 (1).png'
Ix=cv2.imread(questionado, cv2.IMREAD_GRAYSCALE)
Noisex = Ft.NoiseExtractFromImage(questionado, sigma=2.)
Noisex = Fu.WienerInDFT(Noisex, np.std(Noisex))

In [ ]:
# Verifica se as imagens têm o mesmo tamanho
if Noisex.shape != Fingerprint.shape:
    raise ValueError("A imagem e o Fingerprint possuem tamahos distintos.")

# Define o tamanho do bloco
b=32
tamanho_bloco = 2*b+1  # Você pode ajustar este valor conforme necessário
k=50

# Gera o mapa de correlação otimizado
mapa_correlacao = gerar_mapa_correlacao_blocos_paralelo(Noisex, Fingerprint, tamanho_bloco, k, n_jobs=40)

# Normaliza o mapa de correlação para valores entre 0 e 255
mapa_correlacao_norm = (mapa_correlacao * 255).astype(np.uint8)

mapa_correlacao_neg = np.where(mapa_correlacao<0,0,1)



In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(2,2, figsize=(15, 10))
fig.suptitle("Visualização das Imagens", fontsize=16)
axs = axs.ravel()

axs[0].imshow(mapa_correlacao, cmap='jet')
axs[0].set_title('Mapa PRNU')
axs[1].imshow(mapa_correlacao_norm, cmap='jet')
axs[1].set_title('Mapa Truncado')
axs[2].imshow(mapa_correlacao_neg, cmap='jet')
axs[2].set_title('Mapa Negativos')
axs[3].imshow(np.multiply(Ix,mapa_correlacao_neg), cmap='gray')
axs[3].set_title('Sobreposição')
for ax in axs[:1]:
    plt.colorbar(ax.images[0], ax=ax)

# Ajustar o layout e mostrar o plot
#plt.tight_layout()
plt.show()